In [2]:
# Config:

import pandas as pd
from pathlib import Path

MY_NETID = "cdamron3"
TEAMMATE_NETID = "ncash3"

VALIDATION_FILE = Path(f"{MY_NETID}_validation_log.csv")
RESOLUTION_FILE = Path(f"{MY_NETID}_disagreement_resolution.csv")
EXAMPLES_FILE = Path(f"{MY_NETID}_disagreement_examples.md")

df = pd.read_csv(VALIDATION_FILE, keep_default_na=False)

print("Part 1 loaded successfully.")
print(f"Total records: {len(df)}")
print(
    "Disagreements:",
    (df["PreDiscussAgreementstatus(Yes/No)"] == "No").sum()
)


Part 1 loaded successfully.
Total records: 50
Disagreements: 10


In [4]:
# Normalize values to ignore capitalization and extra spaces
def normalize(value):
    return " ".join(str(value).strip().lower().split())

# Update agreement statuses when values match
df.loc[
    df.apply(
        lambda row: normalize(row["AuthorValue"]) ==
                    normalize(row["ValidatorValue"]),
        axis=1
    ),
    "PreDiscussAgreementstatus(Yes/No)"
] = "Yes"

# Save the corrected Part 1 CSV
df.to_csv(VALIDATION_FILE, index=False)

print("Agreement statuses updated!")


Agreement statuses updated!


In [3]:
# Discussion results and Resolution:

discussion_results = {

    # 1. microprediction_timemachines
    ("microprediction_timemachines", "WhyRecovered"): {
        "adjudicated": (
            "Maintainer returned to fix CI workflows, "
            "refresh docs, and update mds"
        ),
        "resolution_type": "ValidatorKept",
        "comments": (
            "The author agreed with the validator's more detailed "
            "explanation of recovery, including CI fixes, "
            "documentation updates, and mds updates."
        )
    },

    # 2. ntustison_crosslong
    ("ntustison_crosslong", "ActivityPattern"): {
        "adjudicated": "Stopped",
        "resolution_type": "ValidatorKept",
        "comments": (
            "The author agreed that the extended inactivity "
            "and minimal subsequent activity support Stopped "
            "rather than Declining."
        )
    },

    # 3. patwie_cppnumericalsolvers
    ("patwie_cppnumericalsolvers", "ActivityPattern"): {
        "adjudicated": "Irregular",
        "resolution_type": "Compromise",
        "comments": (
            "The author initially preferred U-shaped, citing "
            "99 commits in 2016, zero in 2023, and 40 in 2025. "
            "The validator preferred Steady. Both agreed on "
            "Irregular as a compromise because the project "
            "had 14 gaps lasting at least three months and "
            "substantial fluctuations in activity."
        )
    },

    # 4. simplenlg_simplenlg
    ("simplenlg_simplenlg", "ActivityPattern"): {
        "adjudicated": "Declining",
        "resolution_type": "ValidatorKept",
        "comments": (
            "The author agreed with the validator that the "
            "overall commit activity is better described "
            "as Declining rather than Irregular."
        )
    },

    # 5. smtorg_smt
    ("smtorg_smt", "ActivityPattern"): {
        "adjudicated": "Rising",
        "resolution_type": "ValidatorKept",
        "comments": (
            "The author agreed with the validator that "
            "the overall activity trend is Rising "
            "rather than Steady."
        )
    },

    # 6. stan-dev_projpred
    ("stan-dev_projpred", "ActivityPattern"): {
        "adjudicated": "Steady",
        "resolution_type": "ValidatorKept",
        "comments": (
            "The author agreed with the validator that "
            "the project's activity is better described "
            "as Steady rather than Declining."
        )
    },

    # 7. afeinstein20_eleanor
    ("afeinstein20_eleanor", "RecentThemes"): {
        "adjudicated": "Bug Fixes, Merges",
        "resolution_type": "ValidatorKept",
        "comments": (
            "The author agreed with the validator's "
            "classification of recent commits as "
            "Bug Fixes and Merges."
        )
    },

    # 8. algebraicjulia_catlab.jl
    ("algebraicjulia_catlab.jl", "RecentThemes"): {
        "adjudicated": "Dependency Updates, Merges",
        "resolution_type": "ValidatorKept",
        "comments": (
            "The author agreed with the validator that "
            "Dependency Updates and Merges better represent "
            "the recent commits."
        )
    },

    # 9. ntustison_crosslong
    ("ntustison_crosslong", "RecentThemes"): {
        "adjudicated": "Other",
        "resolution_type": "AuthorKept",
        "comments": (
            "The author explained that the latest GitHub "
            "commit was 'Change to private', which fits Other. "
            "The validator's Documentation Updates classification "
            "came from the latest WoC commit, 'Update README.md'. "
            "The Coding Guide specifies GitHub's latest commits, "
            "so the validator accepted Other."
        )
    },

    # 10. patwie_cppnumericalsolvers
    ("patwie_cppnumericalsolvers", "RecentThemes"): {
        "adjudicated": "Other: Bug fixes",
        "resolution_type": "AuthorKept",
        "comments": (
            "The author explained that approximately six of "
            "the ten latest commits concerned CI, tests, "
            "performance, and macOS; three concerned fixes; "
            "and only one concerned Eigen 5 compatibility. "
            "The validator accepted Other and Bug Fixes "
            "as the dominant themes."
        )
    }
}

print(f"Discussion results recorded: {len(discussion_results)}")


Discussion results recorded: 10


In [4]:
# Updateing Validation Log from P1:

actual_disagreements = set(
    zip(
        df.loc[
            df["PreDiscussAgreementstatus(Yes/No)"] == "No",
            "WOCProjectID"
        ],
        df.loc[
            df["PreDiscussAgreementstatus(Yes/No)"] == "No",
            "Field"
        ]
    )
)

expected_disagreements = set(discussion_results)

if actual_disagreements != expected_disagreements:
    raise ValueError(
        "The disagreements in your CSV do not match "
        "the discussion results.\n"
        f"Missing decisions: {actual_disagreements - expected_disagreements}\n"
        f"Unexpected decisions: {expected_disagreements - actual_disagreements}"
    )

for (project, field), result in discussion_results.items():

    mask = (
        (df["WOCProjectID"] == project) &
        (df["Field"] == field)
    )

    df.loc[mask, "AdjudicatedValue"] = result["adjudicated"]
    df.loc[mask, "Comments"] = result["comments"]

# Saving:
df.to_csv(VALIDATION_FILE, index=False)

print(f"Updated: {VALIDATION_FILE}")
print("Adjudicated values and comments added.")

display(
    df.loc[
        df["PreDiscussAgreementstatus(Yes/No)"] == "No",
        [
            "WOCProjectID",
            "Field",
            "AuthorValue",
            "ValidatorValue",
            "AdjudicatedValue"
        ]
    ]
)


Updated: cdamron3_validation_log.csv
Adjudicated values and comments added.


,WOCProjectID,Field,AuthorValue,ValidatorValue,AdjudicatedValue
12,microprediction_timemachines,WhyRecovered,Maintainer returned to fix CI workflows and re...,"Maintainer returned to fix CI workflows, refre...","Maintainer returned to fix CI workflows, refre..."
23,ntustison_crosslong,ActivityPattern,declining,Stopped,Stopped
24,patwie_cppnumericalsolvers,ActivityPattern,U-shaped,Steady,Irregular
26,simplenlg_simplenlg,ActivityPattern,irregular,Declining,Declining
27,smtorg_smt,ActivityPattern,steady,Rising,Rising
28,stan-dev_projpred,ActivityPattern,declining,Steady,Steady
40,afeinstein20_eleanor,RecentThemes,Bug fixes: Other,"Bug Fixes, Merges","Bug Fixes, Merges"
41,algebraicjulia_catlab.jl,RecentThemes,Dependency updates:Other,"Dependency Updates, Merges","Dependency Updates, Merges"
43,ntustison_crosslong,RecentThemes,Other,Documentation Updates,Other
44,patwie_cppnumericalsolvers,RecentThemes,Other: Bug fixes,"Bug Fixes, Dependency Updates",Other: Bug fixes


In [5]:
# Creating CSV:

resolution_df = df[
    df["PreDiscussAgreementstatus(Yes/No)"] == "No"
].copy()

resolution_type_col = (
    "ResolutionType(AuthorKept ValidatorKept/Compromise)"
)

post_agreement_col = (
    "PostDiscussAgreementstatus(Yes/No)"
)

resolution_df[resolution_type_col] = ""
resolution_df[post_agreement_col] = ""

for (project, field), result in discussion_results.items():

    mask = (
        (resolution_df["WOCProjectID"] == project) &
        (resolution_df["Field"] == field)
    )

    resolution_df.loc[
        mask, resolution_type_col
    ] = result["resolution_type"]

    resolution_df.loc[
        mask, post_agreement_col
    ] = "Yes"

resolution_columns = [
    "WOCProjectID",
    "ProjectOwnerNetID",
    "ValidatorNetID",
    "Field",
    "AuthorValue",
    "ValidatorValue",
    "AdjudicatedValue",
    resolution_type_col,
    post_agreement_col,
    "KeyEvidence",
    "Comments"
]

resolution_df = resolution_df[resolution_columns]

resolution_df.to_csv(RESOLUTION_FILE, index=False)

print(f"Created: {RESOLUTION_FILE}")
print(f"Total resolved disagreements: {len(resolution_df)}")

display(resolution_df)


Created: cdamron3_disagreement_resolution.csv
Total resolved disagreements: 10


,WOCProjectID,ProjectOwnerNetID,ValidatorNetID,Field,AuthorValue,ValidatorValue,AdjudicatedValue,ResolutionType(AuthorKept ValidatorKept/Compromise),PostDiscussAgreementstatus(Yes/No),KeyEvidence,Comments
12,microprediction_timemachines,ncash3,cdamron3,WhyRecovered,Maintainer returned to fix CI workflows and re...,"Maintainer returned to fix CI workflows, refre...","Maintainer returned to fix CI workflows, refre...",ValidatorKept,Yes,Longest gap: 2023-07 to 2024-09 (15 months). C...,The author agreed with the validator's more de...
23,ntustison_crosslong,ncash3,cdamron3,ActivityPattern,declining,Stopped,Stopped,ValidatorKept,Yes,Total commits: 311. Timeline: 2016-06-29 to 20...,The author agreed that the extended inactivity...
24,patwie_cppnumericalsolvers,ncash3,cdamron3,ActivityPattern,U-shaped,Steady,Irregular,Compromise,Yes,Total commits: 335. Timeline: 2014-05-29 to 20...,"The author initially preferred U-shaped, citin..."
26,simplenlg_simplenlg,ncash3,cdamron3,ActivityPattern,irregular,Declining,Declining,ValidatorKept,Yes,Total commits: 136. Timeline: 2014-08-20 to 20...,The author agreed with the validator that the ...
27,smtorg_smt,ncash3,cdamron3,ActivityPattern,steady,Rising,Rising,ValidatorKept,Yes,Total commits: 3821. Timeline: 2016-11-08 to 2...,The author agreed with the validator that the ...
28,stan-dev_projpred,ncash3,cdamron3,ActivityPattern,declining,Steady,Steady,ValidatorKept,Yes,Total commits: 6804. Timeline: 2016-06-16 to 2...,The author agreed with the validator that the ...
40,afeinstein20_eleanor,ncash3,cdamron3,RecentThemes,Bug fixes: Other,"Bug Fixes, Merges","Bug Fixes, Merges",ValidatorKept,Yes,Recent commit messages: 2025-08-26: Merge dc5c...,The author agreed with the validator's classif...
41,algebraicjulia_catlab.jl,ncash3,cdamron3,RecentThemes,Dependency updates:Other,"Dependency Updates, Merges","Dependency Updates, Merges",ValidatorKept,Yes,Recent commit messages: 2025-09-29: FEEDBACK: ...,The author agreed with the validator that Depe...
43,ntustison_crosslong,ncash3,cdamron3,RecentThemes,Other,Documentation Updates,Other,AuthorKept,Yes,Recent commit messages: 2025-07-16: Update REA...,The author explained that the latest GitHub co...
44,patwie_cppnumericalsolvers,ncash3,cdamron3,RecentThemes,Other: Bug fixes,"Bug Fixes, Dependency Updates",Other: Bug fixes,AuthorKept,Yes,Recent commit messages: 2025-07-27: Add CITATI...,The author explained that approximately six of...


In [8]:
# Create the 3 Specific Examples of Disagreements
    # Done so I don't have to manually write all that.

example_keys = [
    ("patwie_cppnumericalsolvers", "ActivityPattern"),
    ("ntustison_crosslong", "RecentThemes"),
    ("patwie_cppnumericalsolvers", "RecentThemes")
]

markdown = f"""# MiniProject3 Disagreement Examples

**Project Owner:** {TEAMMATE_NETID}
**Validator:** {MY_NETID}

This document describes three disagreements identified during
the validation of MiniProject2 classifications.

"""

for i, (project, field) in enumerate(example_keys, start=1):

    row = resolution_df[
        (resolution_df["WOCProjectID"] == project) &
        (resolution_df["Field"] == field)
    ].iloc[0]

    markdown += f"""
## Example {i}: {project}

**Field:** {field}

**Parties of Discussion:**
- {TEAMMATE_NETID} (Project Owner)
- {MY_NETID} (Validator)

**AuthorValue:** {row["AuthorValue"]}

**ValidatorValue:** {row["ValidatorValue"]}

**AdjudicatedValue:** {row["AdjudicatedValue"]}

**ResolutionType:** {row[resolution_type_col]}

### Evidence and Discussion

[Write your explanation of the evidence and discussion here.]

### Final Resolution

[Explain why both parties agreed on the adjudicated value.]

---

"""

EXAMPLES_FILE.write_text(markdown, encoding="utf-8")

print(f"Created: {EXAMPLES_FILE}")
print(f"Examples included: {len(example_keys)}")


Created: cdamron3_disagreement_examples.md
Examples included: 3


In [9]:
# Bunch of checks to make sure I did everything right:

assert len(df) == 50, "Validation log must contain 50 rows."

assert len(resolution_df) == 10, (
    "Expected 10 disagreements."
)

assert resolution_df["AdjudicatedValue"].ne("").all(), (
    "Some adjudicated values are missing."
)

assert resolution_df["Comments"].ne("").all(), (
    "Some discussion comments are missing."
)

assert resolution_df[resolution_type_col].isin(
    ["AuthorKept", "ValidatorKept", "Compromise"]
).all(), "Invalid resolution type found."

assert (
    resolution_df[post_agreement_col] == "Yes"
).all(), "Some disagreements remain unresolved."

assert EXAMPLES_FILE.exists(), (
    "Disagreement examples Markdown file was not created."
)

print("PART 2 VALIDATION COMPLETE")
print("=" * 45)

print(f"Validation records: {len(df)}")
print(f"Resolved disagreements: {len(resolution_df)}")
print(f"Markdown examples: {len(example_keys)}")

print("\nResolution breakdown:")
print(resolution_df[resolution_type_col].value_counts())

print("\nFiles generated:")
print(VALIDATION_FILE)
print(RESOLUTION_FILE)
print(EXAMPLES_FILE)


PART 2 VALIDATION COMPLETE
Validation records: 50
Resolved disagreements: 10
Markdown examples: 3

Resolution breakdown:
ResolutionType(AuthorKept ValidatorKept/Compromise)
ValidatorKept    7
AuthorKept       2
Compromise       1
Name: count, dtype: int64

Files generated:
cdamron3_validation_log.csv
cdamron3_disagreement_resolution.csv
cdamron3_disagreement_examples.md
